In [1]:
# importar bibliotecas
import pandas as pd
import numpy as np
import duckdb
from datetime import timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### **Etapa 1 — Coleta e Entendimento dos Dados**

In [2]:
# ── 1. CARREGAMENTO ─────────────────────────────────────────
df = pd.read_csv(
    r'C:\Projects\cashback project\online_retail_II.csv\online_retail_II.csv',
    encoding="ISO-8859-1",   # encoding padrão desse dataset
    dtype={"Customer ID": str, "Invoice": str, "StockCode": str}
)

print(f"Shape bruto: {df.shape}")
df.head()

Shape bruto: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# ── 2. VISÃO GERAL RÁPIDA ────────────────────────────────────
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   object 
 7   Country      1067371 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 65.1+ MB


- Temos nulos em customer id e em description que precisam ser tratados.

In [4]:
# ── 3. GRANULARIDADE E IDENTIFICADORES ──────────────────────
print("=== Granularidade ===")
print(f"Total de linhas        : {len(df):,}")
print(f"Transações únicas      : {df['Invoice'].nunique():,}")
print(f"Clientes com ID        : {df['Customer ID'].nunique():,}")
print(f"Produtos únicos        : {df['StockCode'].nunique():,}")
print(f"Países únicos          : {df['Country'].nunique():,}")

=== Granularidade ===
Total de linhas        : 1,067,371
Transações únicas      : 53,628
Clientes com ID        : 5,942
Produtos únicos        : 5,305
Países únicos          : 43


- Temos +1M de linhas, referentes a +53k compras de +5k produtos distintos, com quase 6k clientes identificados em mais de 40 países.

In [5]:
# ── 4. JANELA TEMPORAL ───────────────────────────────────────
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("=== Janela Temporal ===")
print(f"Início : {df['InvoiceDate'].min()}")
print(f"Fim    : {df['InvoiceDate'].max()}")
print(f"Período: {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} dias")

=== Janela Temporal ===
Início : 2009-12-01 07:45:00
Fim    : 2011-12-09 12:50:00
Período: 738 dias


- Temos um período de dados com um pouco mais de 2 anos (738 dias).

In [6]:
# ── 5. SANIDADE NOS CAMPOS NUMÉRICOS ─────────────────────────
print("=== Estatísticas de Quantity e Price ===")
print(df[["Quantity", "Price"]].describe().round(2))

# Anomalias a observar: valores negativos (devoluções) e zeros
print("\nQuantity <= 0  :", (df["Quantity"] <= 0).sum())
print("Price == 0:", (df["Price"] == 0).sum())

=== Estatísticas de Quantity e Price ===
         Quantity       Price
count  1067371.00  1067371.00
mean         9.94        4.65
std        172.71      123.55
min     -80995.00   -53594.36
25%          1.00        1.25
50%          3.00        2.10
75%         10.00        4.15
max      80995.00    38970.00

Quantity <= 0  : 22950
Price == 0: 6202


- Temos linhas com quantidades negativas e preços zerados que precisam de tratamento.

### **Etapa 2 — Limpeza (Python / Pandas)**

In [7]:
# ── 1. EXCLUSÃO DE REGISTROS SEM Customer ID

# Registrando o tamanho antes de qualquer filtro
df_raw = df.copy()  # preserva o bruto para referência
shape_inicio = df_raw.shape

df = df.dropna(subset=["Customer ID"]) # remove linhas sem Customer ID

pct_removida = (1 - len(df) / len(df_raw)) * 100
print(f"Registros removidos (sem Customer ID): {len(df_raw) - len(df):,} ({pct_removida:.1f}%)")

Registros removidos (sem Customer ID): 243,007 (22.8%)


- Exclusão de clientes não identificados do dataframe.

In [8]:
# ── 2. REMOÇÃO DE DUPLICATAS EXATAS ──────────────────────────
df = df.drop_duplicates()

print(f"Duplicatas removidas: {shape_inicio[0] - len(df):,}")

Duplicatas removidas: 269,486


- Remoção de linhas duplicadas do dataframe.

In [9]:
# ── 3. FILTRO DE QUANTITY E PRICE ────────────────────────

# Separando devoluções antes de excluir — podem virar análise futura
df_devolucoes = df[df["Quantity"] <= 0].copy()
print(f"Registros de devolução isolados: {len(df_devolucoes):,}")

# Base limpa: apenas transações válidas
df = df[df["Quantity"] > 0]
df = df[df["Price"] > 0]

print(f"Após filtro Quantity > 0 e Price > 0: {len(df):,} linhas")

Registros de devolução isolados: 18,390
Após filtro Quantity > 0 e Price > 0: 779,425 linhas


- Devoluções em `df_devolucoes` não foram simplesmente deletadas. Se em etapas futuras quisermos calcular churn ou taxa de retorno por produto, esse DataFrame já está pronto para cruzar.

In [10]:
# ── 4. FOCO GEOGRÁFICO: UNITED KINGDOM ───────────────────────
print("\n=== Distribuição por país (top 10) ===")
print(df["Country"].value_counts().head(10))

df = df[df["Country"] == "United Kingdom"]
print(f"\nApós filtro UK: {len(df):,} linhas")


=== Distribuição por país (top 10) ===
Country
United Kingdom    700388
Germany            16432
EIRE               15565
France             13511
Netherlands         5085
Spain               3662
Belgium             3055
Switzerland         3005
Portugal            2356
Australia           1789
Name: count, dtype: int64

Após filtro UK: 700,388 linhas


- Filtro UK — concentra ~90% das transações e elimina ruído de comportamento de compra muito diferente (moeda, sazonalidade, cultura). Misturar tudo distorceria as métricas de RFM.

In [11]:
# ── 5. CRIAÇÃO DA COLUNA TotalPrice ──────────────────────────
df["TotalPrice"] = df["Quantity"] * df["Price"]

print(f"\nReceita total (UK): £{df['TotalPrice'].sum():,.2f}")
print(df["TotalPrice"].describe().round(2))


Receita total (UK): £14,389,234.92
count    700388.00
mean         20.54
std         237.75
min           0.00
25%           4.25
50%          10.50
75%          17.85
max      168469.60
Name: TotalPrice, dtype: float64


- Criação da coluna Total Price para melhor ligibilidade e futuros cálculos.

In [12]:
# ── 6. RESUMO FINAL DA LIMPEZA ───────────────────────────────
shape_final = df.shape

print("=" * 45)
print("RESUMO DA LIMPEZA")
print("=" * 45)
print(f"Linhas brutas               : {shape_inicio[0]:>10,}")
print(f"Linhas após limpeza         : {shape_final[0]:>10,}")
print(f"Registros removidos         : {shape_inicio[0] - shape_final[0]:>10,}")
print(f"Redução total               : {(1 - shape_final[0]/shape_inicio[0])*100:>9.1f}%")
print(f"\nClientes únicos (UK)        : {df['Customer ID'].nunique():>10,}")
print(f"Transações únicas (UK)      : {df['Invoice'].nunique():>10,}")
print(f"Janela temporal             : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")

RESUMO DA LIMPEZA
Linhas brutas               :  1,067,371
Linhas após limpeza         :    700,388
Registros removidos         :    366,983
Redução total               :      34.4%

Clientes únicos (UK)        :      5,350
Transações únicas (UK)      :     33,541
Janela temporal             : 2009-12-01 → 2011-12-09


### **Etapa 3 — Feature Engineering (SQL/DuckDB + Python)**

#### **Camada 1 - Agregação e Estrutura Temporal (DuckDB / SQL)**

In [13]:
con = duckdb.connect() # Estabelecer uma conexão com o banco de dados 

In [14]:
query_temporal = f"""
    -- CTE 1: Eleva transações para o nível de pedido
    WITH orders AS (
        SELECT
            "Invoice"                        AS invoice,
            "Customer ID"                    AS customer_id,
            "Country"                        AS country,
            CAST(MIN("InvoiceDate") AS DATE) AS order_date,
            ROUND(SUM("TotalPrice"), 2)      AS order_value
        FROM df
        GROUP BY "Invoice", "Customer ID", "Country"
    ),

    -- CTE 2: Numera os pedidos de cada cliente cronologicamente
    orders_ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY order_date, invoice
            ) AS order_rank
        FROM orders
    ),

    -- CTE 3: Puxa contexto do pedido anterior via LAG
    orders_with_context AS (
        SELECT
            *,
            LAG(order_date)  OVER (PARTITION BY customer_id ORDER BY order_rank) AS prev_order_date,
            LAG(order_value) OVER (PARTITION BY customer_id ORDER BY order_rank) AS prev_order_value,
            DATEDIFF(
                'day',
                LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_rank),
                order_date
            ) AS days_since_last
        FROM orders_ranked
    )

    -- Seleção final com bônus candidato calculado
    SELECT
        invoice,
        customer_id,
        country,
        order_date,
        order_value,
        order_rank,
        prev_order_date,
        prev_order_value,
        days_since_last,
        ROUND(prev_order_value * 0.20, 2) AS prev_bonus_candidate
    FROM orders_with_context
    ORDER BY customer_id, order_rank
"""

df_pedidos = con.execute(query_temporal).df()
df_pedidos.head()

,invoice,customer_id,country,order_date,order_value,order_rank,prev_order_date,prev_order_value,days_since_last,prev_bonus_candidate
0,491725,12346.0,United Kingdom,2009-12-14,45.0,1,NaT,NaN,<NA>,NaN
1,491742,12346.0,United Kingdom,2009-12-14,22.5,2,2009-12-14,45.0,0,9.0
2,491744,12346.0,United Kingdom,2009-12-14,22.5,3,2009-12-14,22.5,0,4.5
3,492718,12346.0,United Kingdom,2009-12-18,22.5,4,2009-12-14,22.5,4,4.5
4,492722,12346.0,United Kingdom,2009-12-18,1.0,5,2009-12-18,22.5,0,4.5


- Agregação dos dados para granularidade de pedido e calculo das features temporais necessárias para aplicar as regras
na camada seguinte.

In [15]:
# sanidade básica para verificar se as window functions funcionaram como esperado
print("=== Verificação pós window functions ===")

print(f"Pedidos totais                 : {len(df_pedidos):,}")
print(f"Clientes únicos                : {df_pedidos['customer_id'].nunique():,}")

print(f"\nPrimeiros pedidos (order=1)    : {(df_pedidos['order_rank'] == 1).sum():,}  ← esperado: igual ao nº de clientes")
print(f"prev_order_date NULL           : {df_pedidos['prev_order_date'].isna().sum():,}  ← esperado: igual ao nº de clientes")
print(f"days_since_last NULL           : {df_pedidos['days_since_last'].isna().sum():,}  ← esperado: igual ao nº de clientes")

=== Verificação pós window functions ===
Pedidos totais                 : 33,541
Clientes únicos                : 5,350

Primeiros pedidos (order=1)    : 5,350  ← esperado: igual ao nº de clientes
prev_order_date NULL           : 5,350  ← esperado: igual ao nº de clientes
days_since_last NULL           : 5,350  ← esperado: igual ao nº de clientes


#### **Camada 2 - Aplicação das Regras de Cashback (Python / Pandas)**

In [16]:
# ── Função principal de simulação ──────────────────────────────────────────────
def simulate_cashback(df: pd.DataFrame, validade: int) -> pd.DataFrame:
    """
    Percorre os pedidos de cada cliente em ordem cronológica,
    mantendo saldo de bônus como estado entre iterações.

    Parâmetros
    ----------
    df       : df_pedidos já ordenado com colunas order_rank e days_since_last
    validade : prazo em dias para expiração do bônus acumulado

    Retorna
    -------
    DataFrame com invoice + colunas de resultado sufixadas pela janela (ex: _30d)
    """

    def process_customer(group: pd.DataFrame) -> pd.DataFrame:
        records = []
        saldo = 0.0                              # estado carregado entre pedidos

        for _, row in group.iterrows():
            val   = row["order_value"]
            days  = row["days_since_last"]       # NaN → primeiro pedido do cliente
            bonus_disponivel = saldo             # snapshot do saldo ANTES de alterar

            # ── Classificação e cálculo do resultado ──────────────────────────
            if pd.isna(days):                    # nenhum pedido anterior
                status          = "primeira_compra"
                bonus_aplicado  = 0.0
                bonus_perdido   = 0.0
                bonus_gerado    = round(val * 0.20, 2)
                saldo           = bonus_gerado

            elif days == 0:                      # mesma data do pedido anterior
                status          = "mesmo_dia"
                bonus_aplicado  = 0.0
                bonus_perdido   = 0.0
                bonus_gerado    = round(val * 0.20, 2)
                saldo          += bonus_gerado   # acumula sem resetar

            elif days > validade:                # saldo expirou
                status          = "expirado"
                bonus_aplicado  = 0.0
                bonus_perdido   = round(saldo, 2)
                bonus_gerado    = round(val * 0.20, 2)
                saldo           = bonus_gerado

            elif val >= saldo * 5:               # compra suficiente p/ resgatar tudo
                status          = "resgatado_total"
                bonus_aplicado  = round(saldo, 2)
                bonus_perdido   = 0.0
                bonus_gerado    = round(val * 0.20, 2)
                saldo           = bonus_gerado

            else:                                # compra pequena → resgate parcial
                status          = "resgatado_parcial"
                bonus_aplicado  = round(val * 0.20, 2)
                bonus_perdido   = round(saldo - bonus_aplicado, 2)
                bonus_gerado    = round(val * 0.20, 2)
                saldo           = bonus_gerado

            records.append({
                "invoice"                           : row["invoice"],
                f"status_{validade}d"               : status,
                f"bonus_disponivel_{validade}d"     : round(bonus_disponivel, 2),
                f"bonus_aplicado_{validade}d"       : round(bonus_aplicado,  2),
                f"bonus_perdido_{validade}d"        : round(bonus_perdido,   2),
                f"bonus_gerado_{validade}d"         : round(bonus_gerado,    2),
            })

        return pd.DataFrame(records)

    return (
        df
        .sort_values(["customer_id", "order_rank"])
        .groupby("customer_id", group_keys=False)
        .apply(process_customer, include_groups=False)
        .reset_index(drop=True)
    )


- Criação da função para simular o cashback, mantendo um saldo de bônus atualizado a cada iteração e classificando cada pedido conforme as regras de negócio.

In [17]:
# ── Executar as 3 simulações e enriquecer df_pedidos ──────────────────────────
JANELAS = [30, 45, 60]
for janela in JANELAS:
    print(f"⏳ Simulando validade {janela} dias...", end=" ")
    sim = simulate_cashback(df_pedidos, validade=janela)
    df_pedidos = df_pedidos.merge(sim, on="invoice", how="left")
    print(f"✓  colunas adicionadas: status_{janela}d | bonus_*_{janela}d")

print(f"\n✅ Feature engineering concluído — shape: {df_pedidos.shape}")
df_pedidos.head(5)

⏳ Simulando validade 30 dias... ✓  colunas adicionadas: status_30d | bonus_*_30d
⏳ Simulando validade 45 dias... ✓  colunas adicionadas: status_45d | bonus_*_45d
⏳ Simulando validade 60 dias... ✓  colunas adicionadas: status_60d | bonus_*_60d

✅ Feature engineering concluído — shape: (33541, 25)


,invoice,customer_id,country,order_date,order_value,order_rank,prev_order_date,prev_order_value,days_since_last,prev_bonus_candidate,...,status_45d,bonus_disponivel_45d,bonus_aplicado_45d,bonus_perdido_45d,bonus_gerado_45d,status_60d,bonus_disponivel_60d,bonus_aplicado_60d,bonus_perdido_60d,bonus_gerado_60d
0,491725,12346.0,United Kingdom,2009-12-14,45.0,1,NaT,NaN,<NA>,NaN,...,primeira_compra,0.0,0.0,0.0,9.0,primeira_compra,0.0,0.0,0.0,9.0
1,491742,12346.0,United Kingdom,2009-12-14,22.5,2,2009-12-14,45.0,0,9.0,...,mesmo_dia,9.0,0.0,0.0,4.5,mesmo_dia,9.0,0.0,0.0,4.5
2,491744,12346.0,United Kingdom,2009-12-14,22.5,3,2009-12-14,22.5,0,4.5,...,mesmo_dia,13.5,0.0,0.0,4.5,mesmo_dia,13.5,0.0,0.0,4.5
3,492718,12346.0,United Kingdom,2009-12-18,22.5,4,2009-12-14,22.5,4,4.5,...,resgatado_parcial,18.0,4.5,13.5,4.5,resgatado_parcial,18.0,4.5,13.5,4.5
4,492722,12346.0,United Kingdom,2009-12-18,1.0,5,2009-12-18,22.5,0,4.5,...,mesmo_dia,4.5,0.0,0.0,0.2,mesmo_dia,4.5,0.0,0.0,0.2


### **Etapa 4 — EDA (Python / Matplotlib + Seaborn)**

#### **Segmentação RFM (base para as análises seguintes)**

In [18]:
# ── Definição de datas de referência ──────────────────────────────────────────
dataset_end = df_pedidos["order_date"].max()
cutoff      = dataset_end - pd.Timedelta(days=60)  # quem entra no RFM
data_ref    = cutoff + pd.Timedelta(days=1)         # âncora da recência

# ── Calcular métricas RFM por cliente ─────────────────────────────────────────
rfm = (
    df_pedidos[df_pedidos["order_date"] <= cutoff]
    .groupby("customer_id")
    .agg(
        recencia   = ("order_date", lambda x: (data_ref - x.max()).days),
        frequencia = ("invoice",    "nunique"),
        monetario  = ("order_value","sum"),
    )
    .reset_index()
)

# ── Classificar cada dimensão em 3 faixas com qcut ───────────────────────────
# Recência: menor = melhor → faixas invertidas
rfm["r_score"] = pd.qcut(rfm["recencia"],   q=3, labels=[3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequencia"], q=3, labels=[1, 2, 3]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetario"],  q=3, labels=[1, 2, 3]).astype(int)

rfm["rfm_score"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

# ── Mapeamento de segmentos ───────────────────────────────────────────────────
def segmentar_rf(row):
    if row["r_score"] == 3 and row["f_score"] >= 2:
        return "Recorrentes"   # ativos e habituais
    elif row["r_score"] >= 2:
        return "Ocasionais"    # intervalo aceitável, sem hábito consolidado
    else:
        return "Dormentes"     # sumidos — maior risco de expiração

rfm["segmento"] = rfm.apply(segmentar_rf, axis=1)

# ── Incorporar segmento na tabela analítica ───────────────────────────────────
df_pedidos = df_pedidos.merge(
    rfm[["customer_id", "r_score", "f_score", "m_score", "rfm_score", "segmento"]],
    on="customer_id",
    how="left"
)

# ── Checagem rápida ───────────────────────────────────────────────────────────
print(f"Clientes segmentados : {rfm.shape[0]:,}")
print(f"Shape df_pedidos     : {df_pedidos.shape}")
print()
print(rfm["segmento"].value_counts().to_string())

Clientes segmentados : 5,036
Shape df_pedidos     : (33541, 30)

segmento
Ocasionais     2078
Dormentes      1679
Recorrentes    1279


#### **Intervalo entre compras por segmento**

1. Onde está a massa em relação à janela de 30 dias?

In [19]:
SEGMENTOS = ["Recorrentes", "Ocasionais", "Dormentes"]
CORES     = {"Recorrentes": "#2ECC71", "Ocasionais": "#F39C12", "Dormentes": "#E74C3C"}
VALIDADE    = 30

# ── Base: apenas pedidos com compra anterior ──────────────────────────────────
df_plot = df_pedidos.dropna(subset=["days_since_last"]).copy()

# ── Fig 1 — Distribuição de days_since_last por segmento ─────────────────────
fig1 = go.Figure()

for seg in SEGMENTOS:
    dados = df_plot.loc[df_plot["segmento"] == seg, "days_since_last"]
    fig1.add_trace(go.Violin(
        x       = dados,
        name    = seg,
        line_color      = CORES[seg],
        fillcolor       = CORES[seg],
        opacity         = 0.6,
        orientation     = "h",
        side            = "positive",
        meanline_visible= True,
        points          = False,
    ))

fig1.add_vline(
    x           = VALIDADE,
    line_dash   = "dash",
    line_color  = "#FFFFFF",
    line_width  = 1.5,
    annotation_text     = f"  Validade {VALIDADE}d",
    annotation_font     = dict(color="#FFFFFF", size=11),
    annotation_position = "top right",
)

fig1.update_layout(
    title       = "Distribuição do intervalo entre compras por segmento RFM",
    xaxis_title = "Dias desde a última compra",
    yaxis_title = "Segmento",
    template    = "plotly_dark",
    height      = 420,
    showlegend  = False,
    xaxis       = dict(range=[0, 200]),
)

fig1.show()

**Recorrentes**: A distribuição está quase inteiramente concentrada antes dos 30 dias, o pico é alto e cai rapidamente. Isso significa que esses clientes naturalmente voltam dentro da janela de validade. Para eles, 30 dias é suficiente.

**Ocasionais**: O pico também ocorre antes dos 30 dias, mas a cauda é longa e plana, se estendendo até os 200 dias. Uma parcela expressiva desses clientes retorna entre 30 e 100 dias. Com a validade atual, eles chegam na loja, o cashback já expirou e o incentivo se perde.

**Dormentes**: A distribuição é a mais achatada e larga de todas, o volume de clientes que retorna após 30 dias é dominante. A linha tracejada de validade corta o início da curva, deixando a maior parte da área à direita. Ou seja, a política de 30 dias é quase irrelevante para esse segmento.

2. Proporção de pedidos com status expirado vs. resgatados por segmento — a expiração é generalizada ou concentrada?

In [20]:
# ── Fig 2 — Proporção expirado vs resgatados por segmento ────────────────────
STATUS_ALVO = ["expirado", "resgatado_total", "resgatado_parcial"]
col_status  = f"status_{VALIDADE}d"

# Filtrar e contar por segmento + status
contagem = (
    df_plot[df_plot[col_status].isin(STATUS_ALVO)]
    .groupby(["segmento", col_status])
    .size()
    .reset_index(name="n")
)

# Calcular proporção dentro de cada segmento
contagem["total_seg"] = contagem.groupby("segmento")["n"].transform("sum")
contagem["pct"]       = (contagem["n"] / contagem["total_seg"] * 100).round(1)

CORES_STATUS = {
    "expirado"          : "#E74C3C",
    "resgatado_total"   : "#2ECC71",
    "resgatado_parcial" : "#F39C12",
}

fig2 = go.Figure()

for status in STATUS_ALVO:
    sub = contagem[contagem[col_status] == status]
    fig2.add_trace(go.Bar(
        x           = sub["segmento"],
        y           = sub["pct"],
        name        = status.replace("_", " ").title(),
        marker_color= CORES_STATUS[status],
        text        = sub["pct"].astype(str) + "%",
        textposition= "inside",
    ))

fig2.update_layout(
    barmode     = "stack",
    title       = f"Proporção de status ({VALIDADE}d) por segmento — expiração vs resgates",
    xaxis_title = "Segmento",
    yaxis_title = "% dos pedidos",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
    yaxis       = dict(range=[0, 100]),
)

fig2.show()

A política de 30 dias está convertendo a maior parte do cashback gerado em custo sem retorno — o benefício foi concedido, mas não gerou recompra. Para Dormentes e Ocasionais, a taxa de expiração é praticamente idêntica (~67%), o que sugere que 30 dias não diferencia esses dois perfis em nada.

#### **Impacto financeiro da expiração por segmento**

In [21]:
col_status  = f"status_{VALIDADE}d"
col_perdido = f"bonus_perdido_{VALIDADE}d"
col_gerado  = f"bonus_gerado_{VALIDADE}d"

# ── Gerado: todos os pedidos (denominador) ────────────────────────────────────
gerado_total  = df_pedidos.groupby("segmento")[col_gerado].sum()

# ── Perdido: apenas expirados (numerador) ─────────────────────────────────────
perdido_total = (
    df_pedidos[df_pedidos[col_status] == "expirado"]
    .groupby("segmento")[col_perdido].sum()
)

# ── Montar perda_seg sem merge intermediário ──────────────────────────────────
perda_seg = pd.DataFrame({
    "gerado_total" : gerado_total,
    "perdido_total": perdido_total,
    "clientes"     : df_pedidos.groupby("segmento")["customer_id"].nunique(),
}).reset_index()

perda_seg["perdido_total"]      = perda_seg["perdido_total"].fillna(0)
perda_seg["razao_perda_pct"]    = (perda_seg["perdido_total"] / perda_seg["gerado_total"] * 100).round(1)
perda_seg["perda_media_cliente"]= (perda_seg["perdido_total"] / perda_seg["clientes"]).round(2)

# ── perda_cliente: necessário para o histograma individual ───────────────────
perdido_cliente = (
    df_pedidos[df_pedidos[col_status] == "expirado"]
    .groupby(["customer_id", "segmento"])
    .agg(total_perdido=(col_perdido, "sum"))
    .reset_index()
)

gerado_cliente = (
    df_pedidos
    .groupby(["customer_id", "segmento"])
    .agg(total_gerado=(col_gerado, "sum"))
    .reset_index()
)

perda_cliente = gerado_cliente.merge(perdido_cliente, on=["customer_id", "segmento"], how="left")
perda_cliente["total_perdido"] = perda_cliente["total_perdido"].fillna(0)
perda_cliente["pct_perda"] = (
    perda_cliente["total_perdido"] / perda_cliente["total_gerado"].replace(0, pd.NA) * 100
).round(1)

In [22]:
# ── Fig 1 — Razão perdido / gerado por segmento ──────────────────────────────
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x               = perda_seg["segmento"],
    y               = perda_seg["razao_perda_pct"],
    marker_color    = [CORES[s] for s in perda_seg["segmento"]],
    text            = perda_seg["razao_perda_pct"].astype(str) + "%",
    textposition    = "outside",
    textfont        = dict(size=13),
))

fig1.update_layout(
    title       = f"Razão bônus perdido / bônus gerado por segmento ({VALIDADE}d)",
    xaxis_title = "Segmento",
    yaxis_title = "% do bônus gerado que foi perdido",
    template    = "plotly_dark",
    height      = 420,
    yaxis       = dict(range=[0, perda_seg["razao_perda_pct"].max() * 1.2]),
    showlegend  = False,
)

fig1.show()

A relação entre pedidos expirados e valor perdido não é proporcional. Ocasionais tem maior perda pelo valor de bonus gerado, sugerindo que eles tem um ticket médio maior, e quando expira, a perda absoluta e relativa é mais pesada.

In [23]:
# ── Fig 2 — Distribuição de pct_perda por cliente, facetado por segmento ─────
fig2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=SEGMENTOS,
    shared_yaxes=True,
)

for i, seg in enumerate(SEGMENTOS, start=1):
    dados = perda_cliente.loc[
        (perda_cliente["segmento"] == seg) & (perda_cliente["pct_perda"].notna()),
        "pct_perda"
    ]
    fig2.add_trace(
        go.Histogram(
            x           = dados,
            nbinsx      = 30,
            marker_color= CORES[seg],
            opacity     = 0.85,
            name        = seg,
            showlegend  = False,
        ),
        row=1, col=i,
    )
    # linha de mediana
    mediana = dados.median()
    fig2.add_vline(
        x                   = mediana,
        line_dash           = "dash",
        line_color          = "#FFFFFF",
        line_width          = 1.2,
        annotation_text     = f"  md {mediana:.0f}%",
        annotation_font     = dict(color="#FFFFFF", size=10),
        row=1, col=i,
    )

fig2.update_layout(
    title       = "Distribuição individual da perda de bônus por cliente e segmento",
    template    = "plotly_dark",
    height      = 420,
    xaxis_title = "% do bônus gerado perdido",
)

for ax in ["xaxis", "xaxis2", "xaxis3"]:
    fig2.update_layout({ax: dict(range=[0, 110])})

fig2.show()

# ── Sumário executivo ─────────────────────────────────────────────────────────
print("=" * 58)
print(f"  Impacto financeiro da expiração — janela {VALIDADE} dias")
print("=" * 58)
for _, row in perda_seg.sort_values("razao_perda_pct", ascending=False).iterrows():
    print(
        f"  {row['segmento']:<14} | "
        f"perdido: R$ {row['perdido_total']:>10,.2f} | "
        f"razão: {row['razao_perda_pct']:>5.1f}% | "
        f"média/cliente: R$ {row['perda_media_cliente']:>7.2f}"
    )
print("=" * 58)

  Impacto financeiro da expiração — janela 30 dias
  Ocasionais     | perdido: R$ 331,574.46 | razão:  45.5% | média/cliente: R$  159.56
  Dormentes      | perdido: R$  81,822.55 | razão:  33.5% | média/cliente: R$   48.73
  Recorrentes    | perdido: R$ 617,573.83 | razão:  33.0% | média/cliente: R$  482.86


- Recorrentes são o segmento "mais protegido" pela política de 30 dias, mas são responsáveis pela maior perda absoluta de valor. Isso acontece porque são 1.279 clientes com ticket e volume de compras maiores, logo, o bônus gerado por cliente é muito superior ao dos outros segmentos.

- Ocasionais tem um pico perto de 0% de clientes que conseguiram resgatar o bonus, e ao longo do gráfico vemos uma distribuição mais para a direita de 50%, com uma mediana em 50% que confirma que metade dos clientes perde mais da metade do bônus que recebe.

- Dormentes  são o segmento com menor perda absoluta e menor média por cliente. Para Dormentes, o problema não é a validade do cashback — é que o cashback sequer funciona como gatilho de reativação. A política de validade é secundária aqui; a questão é de engajamento anterior.

#### **Comparação entre cenários (30 / 45 / 60 dias) por segmento**

In [24]:
# ── Consolidar métricas dos 3 cenários por segmento ───────────────────────────
rows = []

for val in [30, 45, 60]:
    col_status  = f"status_{val}d"
    col_gerado  = f"bonus_gerado_{val}d"
    col_perdido = f"bonus_perdido_{val}d"
    col_aplicado= f"bonus_aplicado_{val}d"

    total_pedidos = df_pedidos.groupby("segmento")[col_gerado].count()
    expirados     = (
        df_pedidos[df_pedidos[col_status] == "expirado"]
        .groupby("segmento")[col_gerado].count()
    )

    gerado_total  = df_pedidos.groupby("segmento")[col_gerado].sum()
    perdido_total = (
        df_pedidos[df_pedidos[col_status] == "expirado"]
        .groupby("segmento")[col_perdido].sum()
    )
    aplicado_total= df_pedidos.groupby("segmento")[col_aplicado].sum()

    temp = pd.DataFrame({
        "total_pedidos" : total_pedidos,
        "expirados"     : expirados,
        "gerado_total"  : gerado_total,
        "perdido_total" : perdido_total,
        "aplicado_total": aplicado_total,
    }).reset_index()

    temp["validade"]        = val
    temp["expirados"]       = temp["expirados"].fillna(0)
    temp["perdido_total"]   = temp["perdido_total"].fillna(0)
    temp["taxa_expiracao"]  = (temp["expirados"] / temp["total_pedidos"] * 100).round(1)
    temp["pct_aplicado"]    = (temp["aplicado_total"] / temp["gerado_total"] * 100).round(1)

    rows.append(temp)

cenarios = pd.concat(rows, ignore_index=True)

# ── Fig 1 — Taxa de expiração por janela e segmento ──────────────────────────
fig1 = go.Figure()

for seg in SEGMENTOS:
    sub = cenarios[cenarios["segmento"] == seg]
    fig1.add_trace(go.Scatter(
        x           = sub["validade"],
        y           = sub["taxa_expiracao"],
        mode        = "lines+markers+text",
        name        = seg,
        line        = dict(color=CORES[seg], width=2.5),
        marker      = dict(size=9),
        text        = sub["taxa_expiracao"].astype(str) + "%",
        textposition= "top center",
        textfont    = dict(size=10),
    ))

fig1.update_layout(
    title       = "Taxa de expiração por janela de validade e segmento",
    xaxis       = dict(tickvals=[30, 45, 60], title="Validade (dias)"),
    yaxis_title = "% de pedidos com bônus expirado",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
)

fig1.show()

- Recorrentes tem a maior inclinação dos três segmentos. Cada extensão da janela gera uma queda expressiva de 30d para 45d cai 11,3 pontos, de 45d para 60d cai mais 7,1 pontos. Faz sentido: clientes que já têm cadência de compra próxima dos 30 dias se beneficiam imediatamente de qualquer extensão.

- Ocasionais Partem do ponto mais alto (47,3%) e mostram queda consistente, mas a curva é menos inclinada que a dos Recorrentes. Mesmo em 60 dias, ainda há 32,3% de expiração, uma queda de 15 pontos em relação aos 30 dias, mas longe de resolver o problema completamente. Isso confirma o que o gráfico 1 mostrava: a cauda de Ocasionais se estende bem além dos 60 dias para parte do segmento.

- Dormentes é a linha mais plana. A redução existe, cai ~9 pontos de 30d para 60d, mas é a menor entre os segmentos. Isso reforça o diagnóstico do gráfico 4: Dormentes têm um problema estrutural de engajamento que a validade do cashback não resolve sozinha.

In [25]:
# ── Fig 2 — % do bônus gerado que foi efetivamente aplicado ──────────────────
fig2 = go.Figure()

for seg in SEGMENTOS:
    sub = cenarios[cenarios["segmento"] == seg]
    fig2.add_trace(go.Scatter(
        x           = sub["validade"],
        y           = sub["pct_aplicado"],
        mode        = "lines+markers+text",
        name        = seg,
        line        = dict(color=CORES[seg], width=2.5),
        marker      = dict(size=9),
        text        = sub["pct_aplicado"].astype(str) + "%",
        textposition= "top center",
        textfont    = dict(size=10),
    ))

fig2.update_layout(
    title       = "% do bônus gerado efetivamente aplicado por janela e segmento",
    xaxis       = dict(tickvals=[30, 45, 60], title="Validade (dias)"),
    yaxis_title = "% do bônus gerado que virou desconto",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
)

fig2.show()

- Recorrentes: Já partem do patamar mais alto e crescem consistentemente. O salto de 30d para 45d é de 8,1 pontos — o maior ganho incremental desse segmento. De 45d para 60d ainda cresce 5 pontos, mas com retorno marginal decrescente.

- Ocasionais: Partem de um patamar mais baixo que os Recorrentes, mas a linha tem inclinação consistente. O crescimento de 30d para 60d é de +10,8 pontos percentuais, o maior ganho absoluto entre os três segmentos.

- Dormentes: A curva cresce, mas parte do menor patamar e mantém a menor inclinação. Mesmo em 60 dias, menos de 1 em cada 5 reais de bônus gerado vira desconto de fato. Estender a validade para Dormentes tem retorno marginal.

### **Etapa 5 — Simulação de Cenários**

#### **Tabela de trade-off agregada**

In [26]:
# ── Consolidar métricas dos 3 cenários por segmento ───────────────────────────
rows = []

for val in [30, 45, 60]:
    col_status  = f"status_{val}d"
    col_gerado  = f"bonus_gerado_{val}d"
    col_perdido = f"bonus_perdido_{val}d"
    col_aplicado= f"bonus_aplicado_{val}d"

    gerado_total  = df_pedidos.groupby("segmento")[col_gerado].sum()
    perdido_total = (
        df_pedidos[df_pedidos[col_status] == "expirado"]
        .groupby("segmento")[col_perdido].sum()
    )
    aplicado_total = df_pedidos.groupby("segmento")[col_aplicado].sum()

    temp = pd.DataFrame({
        "gerado_total"  : gerado_total,
        "perdido_total" : perdido_total,
        "aplicado_total": aplicado_total,
    }).reset_index()

    temp["validade"]        = f"{val}d"
    temp["perdido_total"]   = temp["perdido_total"].fillna(0)
    temp["taxa_expiracao"]  = (temp["perdido_total"]  / temp["gerado_total"] * 100).round(1)
    temp["taxa_resgate"]    = (temp["aplicado_total"] / temp["gerado_total"] * 100).round(1)

    rows.append(temp)

cenarios = pd.concat(rows, ignore_index=True)

# ── Pivot: segmento × (cenário × métrica) ────────────────────────────────────
pivot = cenarios.pivot_table(
    index   = "segmento",
    columns = "validade",
    values  = ["taxa_expiracao", "taxa_resgate", "aplicado_total"],
)

# Achatar MultiIndex de colunas → "taxa_expiracao_30d" etc.
pivot.columns = [f"{met}_{val}" for met, val in pivot.columns]
pivot = pivot.reset_index()

# Ordenar segmentos
ordem = {"Champions": 0, "Em risco": 1, "Inativos": 2}
pivot["_ord"] = pivot["segmento"].map(ordem)
pivot = pivot.sort_values("_ord").drop(columns="_ord").reset_index(drop=True)

# ── Formatar para exibição ────────────────────────────────────────────────────
pivot_fmt = pivot.copy()

for val in ["30d", "45d", "60d"]:
    pivot_fmt[f"taxa_expiracao_{val}"] = pivot_fmt[f"taxa_expiracao_{val}"].map("{:.1f}%".format)
    pivot_fmt[f"taxa_resgate_{val}"]   = pivot_fmt[f"taxa_resgate_{val}"].map("{:.1f}%".format)
    pivot_fmt[f"aplicado_total_{val}"] = pivot_fmt[f"aplicado_total_{val}"].map("R$ {:,.2f}".format)

# Renomear colunas para leitura mais fluida
pivot_fmt.columns = [
    "Segmento",
    "Expiração 30d", "Expiração 45d", "Expiração 60d",
    "Resgate 30d",   "Resgate 45d",   "Resgate 60d",
    "Custo 30d",     "Custo 45d",     "Custo 60d",
]

# Reordenar: agrupar por janela em vez de por métrica
colunas_ordenadas = ["Segmento"] + [
    f"{met} {val}"
    for val in ["30d", "45d", "60d"]
    for met in ["Expiração", "Resgate", "Custo"]
]
pivot_fmt = pivot_fmt[colunas_ordenadas]

print("\n  Tabela de trade-off — Segmento × Cenário\n")
display(pivot_fmt)


  Tabela de trade-off — Segmento × Cenário



,Segmento,Expiração 30d,Resgate 30d,Custo 30d,Expiração 45d,Resgate 45d,Custo 45d,Expiração 60d,Resgate 60d,Custo 60d
0,Dormentes,"R$ 28,689.37",33.5%,11.8%,"R$ 37,474.95",28.1%,15.4%,"R$ 44,708.01",23.8%,18.3%
1,Ocasionais,"R$ 115,594.63",45.5%,15.9%,"R$ 160,622.32",36.4%,22.0%,"R$ 194,963.24",29.8%,26.7%
2,Recorrentes,"R$ 628,983.86",33.0%,33.6%,"R$ 780,372.56",21.5%,41.7%,"R$ 872,355.98",14.7%,46.7%


#### **Cálculo do impacto incremental e eficiencia marginal**

In [27]:
# ── Calcular deltas entre janelas ─────────────────────────────────────────────
deltas_rows = []

for seg in SEGMENTOS:
    base = pivot[pivot["segmento"] == seg].iloc[0]

    for (de, para) in [("30d", "45d"), ("45d", "60d")]:
        salto = f"{de} → {para}"

        delta_expiracao = (base[f"taxa_expiracao_{para}"] - base[f"taxa_expiracao_{de}"]).round(1)
        delta_resgate   = (base[f"taxa_resgate_{para}"]   - base[f"taxa_resgate_{de}"]).round(1)
        delta_custo_abs = (base[f"aplicado_total_{para}"] - base[f"aplicado_total_{de}"]).round(2)

        custo_de        = base[f"aplicado_total_{de}"]
        delta_custo_pct = (delta_custo_abs / custo_de * 100).round(1) if custo_de > 0 else pd.NA

        eficiencia = (
            abs(delta_expiracao) / delta_custo_pct * 100
            if delta_custo_pct and delta_custo_pct > 0
            else pd.NA
        )

        # ── Clientes com expiração em cada janela ─────────────────────────────
        clientes_de   = df_pedidos.loc[
            (df_pedidos[f"status_{de}"]   == "expirado") &
            (df_pedidos["segmento"]        == seg),
            "customer_id"
        ].nunique()

        clientes_para = df_pedidos.loc[
            (df_pedidos[f"status_{para}"] == "expirado") &
            (df_pedidos["segmento"]        == seg),
            "customer_id"
        ].nunique()

        deltas_rows.append({
            "segmento"          : seg,
            "salto"             : salto,
            "delta_expiracao_pp": delta_expiracao,
            "delta_resgate_pp"  : delta_resgate,
            "delta_custo_abs"   : delta_custo_abs,
            "delta_custo_pct"   : delta_custo_pct,
            "eficiencia"        : round(eficiencia, 2) if pd.notna(eficiencia) else pd.NA,
            "clientes_expirando_de"  : clientes_de,
            "clientes_expirando_para": clientes_para,
            "delta_clientes"         : clientes_para - clientes_de,  # negativo = melhora
        })

deltas = pd.DataFrame(deltas_rows)

In [28]:
# ── Fig 1 — Redução da expiração por salto e segmento ────────────────────────
fig1 = go.Figure()

for salto, dash in [("30d → 45d", "solid"), ("45d → 60d", "dash")]:
    sub = deltas[deltas["salto"] == salto]
    fig1.add_trace(go.Bar(
        name            = salto,
        x               = sub["segmento"],
        y               = sub["delta_expiracao_pp"].abs(),
        marker_color    = [CORES[s] for s in sub["segmento"]],
        opacity         = 1.0 if dash == "solid" else 0.5,
        text            = sub["delta_expiracao_pp"].apply(lambda v: f"{v:+.1f} pp"),
        textposition    = "outside",
        textfont        = dict(size=11),
    ))

fig1.update_layout(
    barmode     = "group",
    title       = "Redução da taxa de expiração por salto de janela (pontos percentuais)",
    xaxis_title = "Segmento",
    yaxis_title = "Δ taxa de expiração (pp)",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
)

fig1.show()

In [29]:
# ── Fig 2 — Custo incremental por salto e segmento ───────────────────────────
fig2 = go.Figure()

for salto, opacity in [("30d → 45d", 1.0), ("45d → 60d", 0.5)]:
    sub = deltas[deltas["salto"] == salto]
    fig2.add_trace(go.Bar(
        name            = salto,
        x               = sub["segmento"],
        y               = sub["delta_custo_abs"],
        marker_color    = [CORES[s] for s in sub["segmento"]],
        opacity         = opacity,
        text            = sub["delta_custo_pct"].apply(lambda v: f"+{v:.1f}%"),
        textposition    = "outside",
        textfont        = dict(size=11),
    ))

fig2.update_layout(
    barmode     = "group",
    title       = "Custo incremental do programa por salto de janela (R$)",
    xaxis_title = "Segmento",
    yaxis_title = "Δ custo absoluto (R$)",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
)

fig2.show()

In [30]:
# ── Fig 3 — Eficiência marginal (pp de expiração evitados por % de custo adicional) ──
fig3 = go.Figure()

for salto, opacity in [("30d → 45d", 1.0), ("45d → 60d", 0.5)]:
    sub = deltas[deltas["salto"] == salto]
    fig3.add_trace(go.Bar(
        name            = salto,
        x               = sub["segmento"],
        y               = sub["eficiencia"],
        marker_color    = [CORES[s] for s in sub["segmento"]],
        opacity         = opacity,
        text            = sub["eficiencia"].apply(lambda v: f"{v:.1f}"),
        textposition    = "outside",
    ))

fig3.update_layout(
    barmode     = "group",
    title       = "Eficiência marginal por salto de janela e segmento<br>"
                  "<sup>pp de expiração eliminados por cada 1% de custo adicional</sup>",
    xaxis_title = "Segmento",
    yaxis_title = "Eficiência (pp / % custo)",
    template    = "plotly_dark",
    height      = 420,
    legend      = dict(orientation="h", y=-0.2),
)

fig3.show()

Em todos os segmentos, ir de 45 para 60 dias entrega mais pontos percentuais de redução de expiração por cada 1% de custo adicional do que ir de 30 para 45 dias. Mesmo que o benefício absoluto caia no segundo salto, o custo cai proporcionalmente ainda mais rápido. 

**Recorrentes** — Com 47,7 e 57,6, Recorrentes são de longe o segmento mais eficiente. Cada real investido na extensão da janela rende mais para eles do que para qualquer outro grupo. Isso reforça que a política de validade é a alavanca certa para esse segmento e que 60 dias, que parecia questionável pelo custo absoluto no gráfico 8, se justifica quando analisado pela eficiência marginal.

**Ocasionais** — 23,3 no primeiro salto e 30,8 no segundo. A curva crescente confirma que o investimento em Ocasionais melhora de retorno à medida que a janela aumenta. 60 dias para esse segmento é ainda mais justificável do que parecia.

**Dormentes** — Mesmo com o segundo salto sendo mais eficiente (22,3 vs 17,6), os valores absolutos são os menores entre os segmentos em ambos os casos. O cashback simplesmente não é a alavanca certa para reativar Dormentes, independente da janela ou da eficiência marginal.

#### Retorno marginal por salto de janela e segmento

In [31]:
tabela = deltas.sort_values(["salto", "segmento"]).copy()

def hex_to_rgba(hex_color, alpha=1.0):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

fill_colors = [
    hex_to_rgba(CORES[s], alpha=1.0 if i % 2 == 0 else 0.35)
    for i, s in enumerate(tabela["segmento"])
]

fig = go.Figure(go.Table(
    columnwidth = [120, 110, 110, 100, 100, 100, 130],
    header = dict(
        values     = ["<b>Segmento</b>", "<b>Salto</b>", "<b>Δ Expiração</b>",
                      "<b>Δ Custo %</b>", "<b>Eficiência</b>", "<b>Δ Clientes</b>",
                      "<b>Bônus Recup.</b>"],
        fill_color = "#1f2630",
        font       = dict(color="white", size=12),
        align      = "center",
        height     = 36,
    ),
    cells = dict(
        values = [
            tabela["segmento"],
            tabela["salto"],
            tabela["delta_expiracao_pp"].apply(lambda v: f"{v:+.1f} pp"),
            tabela["delta_custo_pct"].apply(lambda v: f"{v:+.1f}%"),
            tabela["eficiencia"].apply(lambda v: f"{v:.2f}" if pd.notna(v) else "—"),
            tabela["delta_clientes"].apply(lambda v: f"{v:+.0f}"),
            tabela["delta_custo_abs"].apply(lambda v: f"R$ {abs(v):,.2f}"),
        ],
        fill_color = [fill_colors] + [["#111827"] * len(tabela)] * 6,
        font       = dict(color="white", size=11),
        align      = ["left", "center", "center", "center", "center", "center", "right"],
        height     = 30,
    ),
))

fig.update_layout(
    title    = dict(
        text = "Retorno marginal por salto de janela e segmento<br>"
               "<sup>Eficiência = pp de expiração reduzidos por cada 1% de custo adicional  |  "
               "Δ Clientes negativo = melhora  |  Bônus Recup. = bônus que migrou de expirado para aplicado</sup>",
        x    = 0.5,
    ),
    template = "plotly_dark",
    height   = 340,
    margin   = dict(t=90, b=20, l=10, r=10),
)

fig.show()

A tabela consolida as três dimensões da decisão: custo, eficiência e valor recuperado. O salto de maior retorno absoluto é **Recorrentes** 30→45d (R$ 151k recuperados), mas o de maior eficiência marginal é **Recorrentes** 45→60d (57,63 pp/% custo), o que indica que ir além dos 45 dias para esse segmento é financeiramente justificável. Para **Ocasionais**, ambos os saltos têm eficiência crescente e bônus recuperado relevante, reforçando 60 dias como janela ideal. **Dormentes** apresentam o menor retorno em todos os cenários, indicando que a alavanca correta para esse segmento está fora da política de validade.

### **Etapa 6 — Conclusões e Recomendações**

#### A validade de 30 dias do cashback é suficiente para todos os perfis de cliente?
Não. A validade de 30 dias não é suficiente para todos os perfis de cliente.
A política atual foi desenhada implicitamente para o cliente Recorrente. Para Ocasionais e 
Dormentes, ela gera perda sistemática de bônus não por comportamento inadequado do cliente, 
mas por incompatibilidade estrutural entre a janela de validade e o padrão real de compra 
de cada segmento.

#### O que os dados revelam sobre cada segmento?
**Recorrentes:** São o segmento mais engajado e, paradoxalmente, o que sofre a maior perda 
financeira absoluta: R$ 617.573,83 — média de R$ 482,86 por cliente. Mesmo voltando com 
frequência, 34,9% dos pedidos ainda expiram com a janela atual e apenas 33,6% do bônus 
gerado vira desconto efetivo. A extensão da janela é a alavanca certa para esse grupo: 
ir de 30 para 45 dias recupera 48 clientes e R$ 151.388,70 em bônus; o salto seguinte, 
de 45 para 60 dias, recupera mais 83 clientes e R$ 91.983,42 — com a maior eficiência 
marginal de toda a análise (57,6 pp por 1% de custo adicional).

**Ocasionais:** São o segmento com a maior razão de perda relativa: 45,5% de todo o bônus 
gerado é desperdiçado. A distribuição de intervalos entre compras é ampla — parte volta em 
menos de 30 dias, mas a maioria tem cadência entre 30 e 100 dias. A extensão da janela 
gera ganhos consistentes em ambos os saltos: 71 clientes e R$ 45.027,69 recuperados no 
primeiro; 62 clientes e R$ 34.340,92 no segundo — com eficiência crescente (23,3 → 30,8). 
É o grupo onde a mudança de política tem o melhor custo-benefício relativo considerando 
o volume de perda e o custo incremental envolvido.

**Dormentes:** A mediana de perda de bônus em 0% revela que o problema não é a validade 
— é o engajamento anterior. Boa parte do segmento simplesmente não gera nem resgata bônus 
de forma significativa. Mesmo no melhor cenário simulado, o salto 30→45d recupera apenas 
45 clientes e R$ 8.785,58 — o menor retorno absoluto entre todos os segmentos e saltos. 
A extensão da janela tem ROI baixo e o investimento é melhor direcionado para outras 
estratégias.

#### Recomendações por segmento

**Recorrentes → Janela de 60 dias:** O salto 45→60d recupera 83 clientes — o maior número 
de qualquer salto na análise — com a mais alta eficiência marginal (57,6). Se houver 
restrição orçamentária, 45 dias já captura -11,5 pp de expiração e R$ 151k recuperados, 
sendo o mínimo recomendado. Mas 60 dias é a política ótima para esse grupo e se justifica 
tanto pelo volume de bônus envolvido quanto pela eficiência do investimento.

**Ocasionais → Janela de 60 dias:** Maior razão de perda financeira relativa (45,5%), 
eficiência crescente nos dois saltos e custo incremental total de ~R$ 81k para recuperar 
133 clientes e aproximadamente R$ 79k em bônus aplicado. É o segmento onde a mudança de 
política tem o argumento de negócio mais direto: cada real investido na extensão da janela 
retorna mais do que qualquer outra combinação segmento-salto com base de custo comparável.

**Dormentes → Estratégia diferente de validade**: A janela de validade não é a alavanca certa. Algumas alternativas com maior potencial de impacto para esse grupo:
- Campanhas de reativação com incentivo direto (cupom fixo, frete grátis) em vez de cashback com prazo
- Comunicação proativa quando o cashback está próximo de expirar
- Reduzir o investimento em cashback para esse segmento e realocar para Recorrentes e Ocasionais